# Room Booking technical notebook

This notebook explains the technical decisions used for the challenge.

## Stack and architecture

- React, TypeScript, and Vite for the browser workspace.
- .NET 8 / ASP.NET Core minimal API for HTTP, authentication, and composition.
- PostgreSQL, EF Core, and Npgsql for persistence.
- Groq's OpenAI-compatible API behind an internal `IChatModel` contract.
- MSTest, FluentAssertions, Testcontainers, Vitest, and Docker for verification and delivery.

The application is a modular monolith. Domain owns booking rules; Application owns use cases and agent orchestration; Infrastructure implements PostgreSQL and Groq adapters; API exposes the HTTP boundary.

## Domain validation

`BookingPeriod` keeps time rules independent of ASP.NET Core, EF Core, and the LLM. Bookings must use UTC instants, exact 30-minute boundaries, a positive duration, and a maximum duration of three hours.

```csharp
// src/RoomBooking.Domain/Bookings/BookingPeriod.cs
public static TimeSpan SlotDuration { get; } = TimeSpan.FromMinutes(30);
public static TimeSpan MaximumDuration { get; } = TimeSpan.FromHours(3);

public bool Overlaps(BookingPeriod other)
{
    return StartUtc < other.EndUtc && other.StartUtc < EndUtc;
}

private static bool IsAlignedToSlot(DateTimeOffset value)
{
    return value.TimeOfDay.Ticks % SlotDuration.Ticks == 0;
}
```

## Concurrent booking protection

Application performs an availability pre-check to return a clear conflict response. PostgreSQL is the final authority when two requests arrive at the same time.

```sql
-- src/RoomBooking.Infrastructure/Persistence/Migrations/...InitialBookingSchema.cs
ALTER TABLE bookings
ADD CONSTRAINT "EX_bookings_room_period_active"
EXCLUDE USING gist
(room_id WITH =, tstzrange(start_utc, end_utc, '[)') WITH &&)
WHERE (status = 1);
```

`[)` makes the range half-open: a booking ending at 11:30 does not conflict with another booking starting at 11:30.

## Agent execution and security

The authenticated user sends a chat message. `ChatAgentService` builds model context and tool definitions. The model returns a final response or requests a tool. The requested tool calls an Application use case, its structured result goes back to the model, and the model produces the final response.

```text
User → ChatAgentService → IChatModel → typed tool
     → Application use case → repository → PostgreSQL
     → tool result → IChatModel → assistant response
```

The execution is bounded to five model iterations and five total tool calls per request. The five available tools are `create_booking`, `list_available_rooms`, `get_room_schedule`, `list_my_bookings`, and `cancel_booking`.

The model never receives `userId` or `ownerId`. `ICurrentUser` resolves identity from the authenticated HTTP request, so tools cannot impersonate another user and cancellation ownership is checked by Application. State-changing API requests also require an antiforgery token.

```csharp
// src/RoomBooking.Application/Abstractions/Authentication/ICurrentUser.cs
public interface ICurrentUser
{
    bool IsAuthenticated { get; }
    string? UserName { get; }
}

// src/RoomBooking.Application/Agent/ChatAgentService.cs
public const int MaximumIterations = 5;
public const int MaximumToolCalls = 5;
```

## Time-zone boundary

For time-based tools, the model provides business-local times without an offset. The server converts them with the configured business timezone (`America/Montevideo` by default) before invoking the UTC Application use cases. Persisted booking instants remain UTC.

```csharp
// src/RoomBooking.Application/Agent/Tools/BusinessLocalTimeConverter.cs
if (startLocal.Kind != DateTimeKind.Unspecified
    || endLocal.Kind != DateTimeKind.Unspecified
    || businessTimeZone.Value.IsInvalidTime(startLocal)
    || businessTimeZone.Value.IsInvalidTime(endLocal)
    || businessTimeZone.Value.IsAmbiguousTime(startLocal)
    || businessTimeZone.Value.IsAmbiguousTime(endLocal))
{
    return false;
}

startUtc = new DateTimeOffset(
    TimeZoneInfo.ConvertTimeToUtc(startLocal, businessTimeZone.Value));
endUtc = new DateTimeOffset(
    TimeZoneInfo.ConvertTimeToUtc(endLocal, businessTimeZone.Value));
```

## Testing approach

Domain and Application tests cover booking invariants and use cases. API integration tests use PostgreSQL through Testcontainers. Agent tests use a deterministic `FakeChatModel`, so CI never needs a Groq key, Internet access, provider availability, or non-deterministic model output. React tests use Vitest and React Testing Library.

The CI workflow restores locked dependencies, builds backend and frontend, runs all tests, and builds the Docker image.

## React client and delivery

The React client is intentionally thin: it shows rooms and the signed-in user's bookings, then sends changes through chat instead of duplicating booking rules in the browser. Chat effects refresh the workspace after a booking is created or cancelled.

```tsx
// frontend/RoomBooking.Web/src/components/ChatPanel.tsx (flow excerpt)
const response = await chatApi.sendMessage(sessionId, message);
setMessages((current) => [...current, response.assistantMessage]);

if (response.effects.includes('booking_created')) {
  await refreshWorkspace();
}
```

A multi-stage Dockerfile builds React, publishes the API, and copies the static client into `wwwroot`. Production is one same-origin application container plus PostgreSQL. Database migrations are applied manually and never during application startup.